# Test of the src package

Part A checks the data-preparation conventions, the history columns and the ENG-19b derivations
against the committed `results/` files (no raw data, under a minute). Part B re-extracts four cells
from the Severson pickles and compares them with the committed feature tables; it needs the three
pickles (~25 s to load, ~8 GB) and is **off by default**: set `RUN_PART_B = True` in the next cell
and run all.

In [1]:
RUN_PART_B = False  # True: run Part B (loads the three Severson pickles, ~25 s and ~8 GB)

## Setup

In [2]:
import gc
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# run from the repo root (the parent of this notebooks/ folder) so src/ imports resolve
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src" / "config.py").exists(), "open this notebook from the notebooks/ folder"
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import RESULTS_DIR, pickles_available
from src.data_prep import assign_folds, clean_feature_table, split_dev_test
from src.features import FEATURE_COLS, RECOMPUTE_COLS, SECOND_PEAK_COLS, extract_all_features
from src.features_temporal import (DEFAULT_MIN_HISTORY, HISTORY_COL, TEMPORAL_COLS,
                                   add_temporal_features)
from src.peak_extraction import spot_check
from src.severson_loader import load_clean_severson
from src.subset_search import analyse_table, mean_predictor_cv, table_from_records


def read_result(name):
    return json.loads((RESULTS_DIR / name).read_text(encoding="utf-8"))


def assert_close(got, want, tol=1e-12, path=""):
    """Nested dicts/lists equal; floats within an absolute tol, everything else exact."""
    if isinstance(want, dict):
        assert set(got) == set(want), f"{path}: keys {sorted(got)} != {sorted(want)}"
        for k in want:
            assert_close(got[k], want[k], tol, f"{path}.{k}")
    elif isinstance(want, (list, tuple)):
        assert len(got) == len(want), f"{path}: length {len(got)} != {len(want)}"
        for i, (g, w) in enumerate(zip(got, want)):
            assert_close(g, w, tol, f"{path}[{i}]")
    elif isinstance(want, float):
        assert abs(got - want) <= tol, f"{path}: {got!r} != {want!r}"
    else:
        assert got == want, f"{path}: {got!r} != {want!r}"


print("repo root:", REPO_ROOT)

repo root: C:\Users\Brend\dev\fyp-ica-soh-deliverables


## A1. Cleaning counts

In [3]:
feat = pd.read_parquet(RESULTS_DIR / "features_expanded.parquet")
clean, log = clean_feature_table(feat, verbose=True)
assert log == {"start": 100_501, "after_status_ok": 100_459, "after_dropna_soh": 100_376,
               "after_soh_min": 100_335, "after_soh_max": 100_331, "soh_clean_band": [0.5, 1.05]}
assert log == read_result("eng00_baseline.json")["clean_log"]
assert len(clean) == 100_331 and (clean["status"] == "ok").all()
print("A1 PASS: 100,501 -> 100,459 -> 100,376 -> 100,335 -> 100,331 rows, as in eng00_baseline.json")

  clean: 100501 -> ok 100459 -> dropna 100376 -> SoH>0.5 100335 -> SoH<=1.05 100331
A1 PASS: 100,501 -> 100,459 -> 100,376 -> 100,335 -> 100,331 rows, as in eng00_baseline.json


## A2. Dev/test split

In [4]:
dev, test = split_dev_test(clean)
assert dev["cell_id"].nunique() == 84 and test["cell_id"].nunique() == 40
assert len(dev) == 59_131 and len(test) == 41_200
assert set(dev["batch"]) == {"b1", "b2"} and set(test["batch"]) == {"b3"}
assert not set(dev["cell_id"]) & set(test["cell_id"])
print(f"A2 PASS: dev {dev['cell_id'].nunique()} cells / {len(dev):,} rows, "
      f"test {test['cell_id'].nunique()} cells / {len(test):,} rows")

A2 PASS: dev 84 cells / 59,131 rows, test 40 cells / 41,200 rows


## A3. Fold map

In [5]:
fold, cell_to_fold = assign_folds(dev)
assert sorted(fold.unique()) == [0, 1, 2, 3, 4]
assert (fold.groupby(dev["cell_id"]).nunique() == 1).all()  # no cell in two folds
assert (dev["cell_id"].map(cell_to_fold) == fold).all()
assert {c: int(f) for c, f in cell_to_fold.items()} == read_result("eng00_baseline.json")["cell_to_fold"]
per_fold = pd.Series(cell_to_fold).value_counts().sort_index().to_dict()
print(f"A3 PASS: 5 folds, cells per fold {per_fold}, equal to eng00_baseline.json cell_to_fold")

A3 PASS: 5 folds, cells per fold {0: 16, 1: 16, 2: 16, 3: 18, 4: 18}, equal to eng00_baseline.json cell_to_fold


## A4. History gate

In [6]:
hist, history_cols = add_temporal_features(clean)
since_first = hist["cycle"] - hist.groupby("cell_id")["cycle"].transform("min")
expect_nan = since_first < DEFAULT_MIN_HISTORY
t = hist[history_cols]
assert history_cols == TEMPORAL_COLS and len(history_cols) == 54
assert (t.isna().all(axis=1) == expect_nan).all() and (t.isna().any(axis=1) == expect_nan).all()
assert (hist[HISTORY_COL] == ~expect_nan).all()
assert int(hist[HISTORY_COL].sum()) == 97_852
print(f"A4 PASS: {int(hist[HISTORY_COL].sum()):,} rows with history, {len(history_cols)} columns, "
      f"NaN exactly on the first {DEFAULT_MIN_HISTORY} cycles of each cell")

A4 PASS: 97,852 rows with history, 54 columns, NaN exactly on the first 20 cycles of each cell


## A5. Subset-search derivations

In [7]:
# The per-subset rows of the ENG-19 search live in the CSV (the JSON holds only summaries);
# the table is rebuilt as OLD scripts/eng19b_subset_analysis.py did.
records = {}
for r in pd.read_csv(RESULTS_DIR / "eng19_exhaustive_subsets.csv").to_dict("records"):
    records[r["features"]] = {"blocks": r["features"].split("+"), "k": int(r["k"]),
                              "n_columns": int(r["k"]),
                              "cv_pooled_rmse_mean": float(r["cv_pooled_rmse_mean"]),
                              "cv_pooled_rmse_std": float(r["cv_pooled_rmse_std"])}
table = table_from_records(records)
stored = read_result("eng19b_analysis.json")
assert len(table) == stored["n_subsets"] == 511 and stored["names"] == list(FEATURE_COLS)

v_empty = mean_predictor_cv(dev, fold)  # dev-CV RMSE of the training-fold mean
assert_close(v_empty, stored["v_empty"], 1e-12, "v_empty")

analysis = analyse_table(table, list(FEATURE_COLS), stored["v_empty"]["cv_pooled_rmse_mean"])
assert_close(analysis["best_per_k"], stored["best_per_k"], 1e-12, "best_per_k")
assert analysis["be_drop_order"] == stored["be_drop_order"]
assert analysis["be_last_standing"] == stored["be_last_standing"]
print("A5 PASS: v_empty, best_per_k and be_drop_order reproduce eng19b_analysis.json")
print("  backward elimination drops " + " > ".join(stored["be_drop_order"])
      + f"; last standing: {stored['be_last_standing']}")

A5 PASS: v_empty, best_per_k and be_drop_order reproduce eng19b_analysis.json
  backward elimination drops slope_left > area35 > height > area25 > area > slope_right > position > fwhm; last standing: prominence


## Part A result

In [8]:
print("Part A: all five checks passed (A1 cleaning, A2 split, A3 folds, A4 history, A5 subsets)")

Part A: all five checks passed (A1 cleaning, A2 split, A3 folds, A4 history, A5 subsets)


## Part B. Extraction from the pickles

Needs the three Severson pickles under `src.config.PKL_DIR` (~25 s to load, ~8 GB). Off by default:
set `RUN_PART_B = True` in the first code cell. B1 runs `spot_check` on b1c1, b1c2, b1c9 and b1c20
against `results/features_3feat.parquet`; B2 runs `extract_all_features` on b1c1 against
`results/features_expanded.parquet`. The asserts follow the OLD notebook's checks (cells 14, 25, 32).

In [9]:
SPOT_CELLS = ["b1c1", "b1c2", "b1c9", "b1c20"]
if RUN_PART_B:
    assert pickles_available(), "the three Severson pickles are not under src.config.PKL_DIR"
    cells = load_clean_severson(verbose=False)
    print(f"{len(cells)} cells loaded")
else:
    print("Part B skipped: RUN_PART_B = False")

Part B skipped: RUN_PART_B = False


### B1. spot_check vs features_3feat.parquet

In [10]:
if RUN_PART_B:
    want3 = pd.read_parquet(RESULTS_DIR / "features_3feat.parquet")
    got3 = spot_check(cells, SPOT_CELLS)
    cols3 = ["cycle", "height", "position", "area", "SOH", "status"]
    for cid in SPOT_CELLS:
        want = want3.loc[want3["cell_id"] == cid, cols3].reset_index(drop=True)
        got = got3[cid][cols3].reset_index(drop=True)
        pd.testing.assert_frame_equal(got, want, check_dtype=False, check_exact=True)
    print(f"B1 PASS: spot_check on {', '.join(SPOT_CELLS)} reproduces every row of "
          f"features_3feat.parquet exactly ({sum(len(d) for d in got3.values()):,} rows)")

### B2. extract_all_features vs features_expanded.parquet

In [11]:
if RUN_PART_B:
    want9 = feat.loc[feat["cell_id"] == "b1c1"].reset_index(drop=True)
    got9 = extract_all_features(cells["b1c1"], "b1c1", verbose=True)
    exact = ["cycle", "SOH", "status"] + FEATURE_COLS + SECOND_PEAK_COLS
    pd.testing.assert_frame_equal(got9[exact], want9[exact], check_dtype=False, check_exact=True)
    # the recomputed-curve columns are reproduced to < 1e-12 (model: OLD notebook cell 32)
    assert (got9[RECOMPUTE_COLS].isna() == want9[RECOMPUTE_COLS].isna()).all().all()
    diff = (got9[RECOMPUTE_COLS] - want9[RECOMPUTE_COLS]).abs().to_numpy()
    worst = float(np.nanmax(diff)) if np.isfinite(diff).any() else 0.0
    assert worst < 1e-12
    print(f"B2 PASS: {len(got9):,} rows of b1c1; cycle, SOH, status, the nine features and the "
          f"second-peak columns identical; recompute columns within {worst:.1e}")

### B3. Free memory

In [12]:
if RUN_PART_B:
    del cells
    gc.collect()
    print("Part B: both checks passed; pickles released")